# The Lab — Quick Start Notebook

This notebook shows how to use The Lab from Python for exploratory work.

In [ ]:
from thelab.quick import experiment, compare, list_models
from thelab.run.inspect import inspect_dataset, format_inspect

## 1. Inspect a dataset

In [ ]:
print(format_inspect(inspect_dataset("examples/iris.csv", target="species")))

## 2. Train one model

In [ ]:
exp = experiment("examples/iris.csv", target="species", model="logistic_regression")
print(exp)
print(exp.metrics)

## 3. Compare all models in-memory

In [ ]:
experiments = compare("examples/iris.csv", target="species")
for e in experiments:
    print(f"{e.model:25} accuracy={e.metrics.get('test_accuracy', 0):.4f}")

## 4. Predict with the best model

In [ ]:
best = max(experiments, key=lambda e: e.metrics.get("test_accuracy", 0))
print("Best:", best.model)
print("Prediction:", best.predict([5.1, 3.5, 1.4, 0.2]))

## 5. Dry-run a model quickly

In [ ]:
from thelab.quick import experiment

quick = experiment("examples/wine.csv", target="class", model="random_forest", dry_run=True)
print(quick.metrics)

## 4. Real-dataset cleaning (P2)

The deterministic cleaning policy parses datetimes, one-hot encodes low-cardinality categoricals, frequency-encodes high-cardinality ones, and returns an audit report.

In [ ]:
from pathlib import Path
from thelab.ide.cleaning import clean_dataset

uploads = sorted(Path("data/uploads").glob("*.csv"))
if not uploads:
    print("Upload a CSV via the dashboard (or copy one to data/uploads/) first.")
else:
    dataset_id = f"uploads/{uploads[0].name}"
    target = "accuracy_90d"  # <-- set your target column
    result = clean_dataset(dataset_id, target=target)
    for action in result["cleaning_report"]["actions"]:
        print("-", action)
    print(result["dataset_id"], f"{result['rows']} rows x {result['columns']} cols")

## 5. Agent proposals without an LLM

The WorkerAgent falls back to a deterministic, EDA-grounded proposal when no provider is configured.

In [ ]:
import asyncio
from thelab.agents.mock import MockProvider
from thelab.agents.worker import WorkerAgent

worker = WorkerAgent(provider=MockProvider([]), servers=[], proposals_dir="proposals")
proposal = asyncio.run(
    worker.propose(goal="Predict the target",
                   dataset="examples/iris.csv", target="species",
                   model_grid=["random_forest"], seeds=[42])
)
print(proposal.proposal_id, proposal.model_grid, proposal.seeds)